# FLARE / ACE-Grade — Adversarial Cross-Examination for Assessment

Same three-round debate mechanism as the wildfire-tweet demo, applied to grading open-ended short-answer responses instead of labelling social media relevance. Two independent LLMs — Claude, GPT — grade an answer separately, then cross-examine each other's reasoning before a verdict is finalized. Disagreement routes the case to a human instructor rather than being averaged away.

## Setup

Model verdicts on contestable answers vary between calls (neither model accepts a temperature override). Cases 1–4 replay real captured transcripts from `demo_cache/pinned/`; the bonus cell at the end calls both APIs live.

In [ ]:
from ace_grade import ACEGrade, GradeOutcome
from IPython.display import Markdown, display

engine = ACEGrade()

OUTCOME_BADGE = {
    "agree":     "✅ **AGREE**",
    "converged": "⚠️ **CONVERGED**",
    "disagree":  "🔺 **DISAGREE — escalated to instructor**",
}

def display_result(result):
    lines = []
    lines.append("**Round 1 — independent grading**")
    lines.append(f"- Claude: `{result.round1_claude.verdict.value}` — {result.round1_claude.reasoning}")
    lines.append(f"- GPT: `{result.round1_gpt.verdict.value}` — {result.round1_gpt.reasoning}")

    if result.outcome != GradeOutcome.AGREE:
        lines.append("")
        lines.append("**Round 2 — adversarial flaw-finding**")
        lines.append("- Claude found in GPT's reasoning:")
        for f in result.round2_claude_flaw.flaws_found:
            lines.append(f"  - {f}")
        lines.append("- GPT found in Claude's reasoning:")
        for f in result.round2_gpt_flaw.flaws_found:
            lines.append(f"  - {f}")
        lines.append("")
        lines.append("**Round 3 — rebuttal**")
        lines.append(f"- Claude: `{result.round3_claude.revised_verdict.value}` (changed mind: {result.round3_claude.changed_mind})")
        lines.append(f"- GPT: `{result.round3_gpt.revised_verdict.value}` (changed mind: {result.round3_gpt.changed_mind})")

    lines.append("")
    badge = OUTCOME_BADGE[result.outcome.value]
    final = result.final_verdict.value if result.final_verdict else "None — instructor decides"
    lines.append(f"**Outcome:** {badge}  ")
    lines.append(f"**Final verdict:** `{final}`  ")
    lines.append(f"**Confidence:** {result.confidence:.0%}")

    if result.instructor_escalation_summary:
        lines.append("")
        lines.append("---")
        lines.append("```")
        lines.append(result.instructor_escalation_summary)
        lines.append("```")

    display(Markdown("\n\n".join(lines)))

## The rubric question

Grading a short-answer response about *why LLMs hallucinate* — a deliberately meta topic for this audience.

In [ ]:
QUESTION = (
    "Explain why a language model can produce a fluent, confident-sounding "
    "answer that is factually wrong (hallucination), and describe one "
    "mitigation strategy."
)

RUBRIC = """
A strong answer must:
1. Correctly explain the mechanism: LLMs generate the statistically most probable
   next token given context, not a verified fact lookup -- fluency and factual
   correctness are not the same optimization target, so confident phrasing does
   not imply verified truth.
2. Describe at least one concrete, correctly-explained mitigation (e.g.
   retrieval-augmented generation grounding claims in retrieved documents,
   output verification/self-consistency checks, human-in-the-loop review) --
   not just naming a technique without explaining how it addresses (1).

Partial credit: correct on one dimension but not the other, or uses correct
vocabulary without demonstrating genuine mechanistic understanding.

Does not meet: mechanism explanation is absent or incorrect, or the mitigation
does not actually address the model-side cause of hallucination.
"""

display(Markdown(f"**Question:** {QUESTION}\n\n**Rubric:**\n```\n{RUBRIC}\n```"))

## Case 1 — A genuinely strong answer

Correct mechanism, correct mitigation, clearly explained. Expect fast agreement.

In [ ]:
answer_case1 = (
    "LLMs are trained to predict the most probable next token given the "
    "preceding context, not to verify claims against a ground-truth knowledge "
    "base. Because fluency and factual grounding are not the same objective, "
    "the model can produce grammatically confident text even when the "
    "underlying claim is fabricated. One mitigation is retrieval-augmented "
    "generation: the system retrieves relevant documents at inference time and "
    "conditions the answer on that retrieved evidence, grounding claims in "
    "checkable text instead of relying solely on the model's internal recall."
)

result1 = engine.grade_from_pinned_transcript(QUESTION, answer_case1, case_id="case1_strong")
display_result(result1)

*Both models land on `meets_criteria` in Round 1 — no cross-examination triggered.*

## Case 2 — Fluent but hollow

Correct-sounding vocabulary, no real mechanism explained — the case a keyword-matching grader would wave through.

In [ ]:
answer_case2 = (
    "Hallucination happens because language models sometimes generate "
    "incorrect information due to the probabilistic nature of how they work. "
    "This is a well-known limitation of AI systems. A good way to mitigate "
    "hallucination is to use RAG, which helps reduce hallucinations by "
    "grounding the model. RAG is a popular technique that many companies use "
    "to make their AI more reliable and trustworthy. Overall, hallucination "
    "is an important challenge in AI that researchers are actively working "
    "to solve."
)

result2 = engine.grade_from_pinned_transcript(QUESTION, answer_case2, case_id="case2_hollow")
display_result(result2)

*Claude and GPT split in Round 1 (`does_not_meet` vs `partially_meets`); after cross-examination both converge on `partially_meets`, at lower confidence — flagged for optional review, not auto-approved or escalated.*

## Case 3 — A deliberately planted error

A specific, real misconception, wrong by construction — a free calibration check with no gold-labelled data needed.

In [ ]:
answer_case3 = (
    "Language models hallucinate because they intentionally make up "
    "information whenever they don't know an answer, similar to a search "
    "engine returning no results -- the model is essentially guessing "
    "randomly when its training data doesn't cover a topic. This means "
    "hallucinations happen only on rare or obscure topics that weren't in "
    "the training data. The best mitigation is simply increasing the size "
    "of the training dataset so the model has seen more facts and doesn't "
    "need to guess as often."
)

result3 = engine.grade_from_pinned_transcript(QUESTION, answer_case3, case_id="case3_planted_error")
display_result(result3)

caught = result3.final_verdict is not None and result3.final_verdict.value == "does_not_meet"
display(Markdown(f"### Calibration check: planted error {'✅ CAUGHT' if caught else '❌ MISSED'}"))

*Both models catch the planted error independently and unanimously.*

## Case 4 — The case that actually needs debate

Same concept, explained entirely through analogy instead of technical vocabulary. A genuine judgment call: does correct understanding without the vocabulary deserve full credit?

In [ ]:
answer_case4 = (
    "It's kind of like a student writing an exam answer purely from memory "
    "without ever checking a textbook. If the student is confident and writes "
    "clearly, the answer will read as convincing even if a detail is "
    "misremembered -- the writing quality gives no signal about whether the "
    "content itself was double-checked. The same is true for these models: "
    "they write fluently based on what they've learned, not by looking "
    "anything up in the moment, so confident phrasing doesn't guarantee the "
    "facts are right. One way to fix this is to let the model actually 'look "
    "things up' before answering, similar to letting the student check the "
    "textbook before submitting the exam, so the answer is grounded in "
    "something outside its memory instead of just what it recalls."
)

result4 = engine.grade_from_pinned_transcript(QUESTION, answer_case4, case_id="case4_analogy")
display_result(result4)

*Claude and GPT split in Round 1, then swap positions after cross-examination — still disagree after Round 3, so it escalates to the instructor with both full arguments attached.*

## Bonus — a live call

Everything above replayed captured transcripts. This cell calls the real Claude + GPT APIs live — edit `live_answer` and re-run with any text.

In [ ]:
live_answer = (
    "LLMs hallucinate because they generate the next most likely word based "
    "on patterns in their training data, not because they check facts. So a "
    "fluent answer can still be wrong. One mitigation is retrieval-augmented "
    "generation, which grounds the answer in retrieved documents."
)

result_live = engine.grade(QUESTION, RUBRIC, live_answer, case_id="live_demo")
display_result(result_live)